In [121]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [122]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [123]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [124]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [125]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [127]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [128]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [129]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 115,
 'tn': 2596,
 'fp': 41,
 'fn': 248,
 'misclassification_rate': 0.09633333333333334,
 'false_positive_rate': 0.015547971179370497,
 'false_negative_rate': 0.6831955922865014}

### Check results on the test set (new data not yet seen by the model)

In [130]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 35,
 'tn': 859,
 'fp': 15,
 'fn': 91,
 'misclassification_rate': 0.106,
 'false_positive_rate': 0.017162471395881007,
 'false_negative_rate': 0.7222222222222222}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

My model has a misclassification rate of 10.6% on the test data, meaning that it correctly classified about 89.4% of the observations. This gives me some confidence in the model overall. However, I would not be highly confident in its ability to specifically identify bots because the false negative rate is still relatively high. The model correctly identified 35 bots but incorrectly classified 91 bots as humans. Therefore, the overall misclassification rate is fairly low, the model is much better at identifying humans than it is at identifying bots.

### What are potential ramifications of false positives from the model?

A false positive occurs when the model identifies a real human as a bot. This could cause legit users to be incorrectly flagged, restricted, or possibly banned from the platform. It could also ask users to complete additional verification and create a frustrating experience for people who have done nothing wrong. In my test results, there were 15 false positives.

### What are potential ramifications of false negatives from the model?

A false negative occurs when an actual bot is incorrectly classified as a human. This would allow bots to avoid detection and continue operating on the platform. Depending on the purpose of the bots, they could continue activities such as sending spam, posting misleading content, etc. This is an issue for my model because it produced 91 false negatives, which is much higher than the number of false positives.